In [ ]:
import pandas as pd
import json
import unicodedata
from pathlib import Path

hotels_path = Path("hotels/hotels.csv")
if not hotels_path.exists():
    hotels_path = Path("../hotels/hotels.csv")

if not hotels_path.exists():
    raise FileNotFoundError(
        f"Could not find hotels.csv. Checked: {hotels_path.resolve()}"
    )

hotels = pd.read_csv(hotels_path, encoding="utf-8")
with open("cities.json", "r", encoding="utf-8") as file:
    cities = json.load(file)

cities_df = pd.DataFrame(cities)

def normalize_city(city):
    if pd.isna(city):
        return ""
  
    city = str(city).lower().strip()
    city = unicodedata.normalize("NFKD", city)
    city = "".join(c for c in city if not unicodedata.combining(c))

    return city

hotels["city_normalized"] = hotels["city"].apply(normalize_city)
cities_df["city_normalized"] = cities_df["name"].apply(normalize_city)

city_map = dict(zip(cities_df["city_normalized"], cities_df["id"]))

hotels["city_id"] = hotels["city_normalized"].map(city_map)

matched = hotels[hotels["city_id"].notna()].copy()
unmatched = hotels[hotels["city_id"].isna()].copy()

print("========== CITY MATCHING ==========")
print("Total hotel records:", len(hotels))
print("Matched records:", len(matched))
print("Unmatched records:", len(unmatched))

print("\nTotal hotel cities:", hotels["city_normalized"].nunique())
print("Matched cities:", matched["city_normalized"].nunique())
print("Unmatched cities:", unmatched["city_normalized"].nunique())

print("\n========== UNMATCHED CITIES ==========")
print(unmatched["city"].value_counts().head(30))

matched.to_csv("matched_hotels.csv", index=False)

print("\nmatched_hotels.csv created")

========== CITY MATCHING ==========
Total hotel records: 6000
Matched records: 3375
Unmatched records: 2625

Total hotel cities: 543
Matched cities: 148
Unmatched cities: 395

========== UNMATCHED CITIES ==========
city
bangalore         193
cochin             89
jaisalmer          79
alleppey           70
calangute          62
ooty               41
dharamshala        41
candolim           38
pushkar            37
kodaikānāl         37
baga               36
palolem            33
lonavala           32
haridwār           31
munnar             31
anjuna             30
thekkady           29
dalhousie          29
bhubaneshwar       28
kumarakom          27
katra              24
kolhapur           22
mandrem            20
darjeeling         20
salt lake city     20
varca              18
gangtok            17
ghaziabad          17
vagator            17
trivandrum         16
Name: count, dtype: int64

matched_hotels.csv created


In [94]:
print("========== UNMATCHED CITY CHECK ==========")

matched_names = set(cities_df["city_normalized"])

unmatched_cities = sorted(
    set(hotels["city_normalized"]) - matched_names
)

print("Unmatched unique cities:", len(unmatched_cities))
print()

for city in unmatched_cities:
    print(city)

========== UNMATCHED CITY CHECK ==========
Unmatched unique cities: 395


abdullapuram
abu road
adur
agonda
aguada
ahar
ahmadnagar
ajanta
alibaug
alipur
alleppey
almora
alto porvorim
ambalapulai
ambalavayal
angamali
anjuna
arambol
arpora
attibele
attigundi
auroville
avathi
badami
baddi
badshahpur
baga
baghdogra
bajaura
balighai
ballygunge
bambolim
banar
bangalore
bapane
bara bazar
barddhaman
bardez
bareilly
barkur
basari
bashist
bayappanhalli
beawar
begur
behala
behror
belgaum
belparao
belur
benaulim
betalbatim
bhatkal
bhilai
bhim tal
bhimadolu
bhimashankar
bhiwadi
bhubaneshwar
bhurtuk
bijni
binsar
bir
bodh gaya
bolpur
bordi
bori
buin
bulandshahr
bundi
calangute
canacona
candolim
carmona
cavelossim
chakan
chamba
chandoli
chandrapur
chegat
cherai beach
chhapar
chikalthan
chikmagalur
chinchwad
chinnakanal
chitradurga
chopdem
chottanikara
churi jodhan
churu
cochin
collem
colva
coonoor
curca
dagshai
dahej
dalhousie
dam dam
daman
dapoli
darjeeling
daulatabad
davangere
dedha
delmara
delwara


In [95]:
import pandas as pd
import json
import unicodedata
from rapidfuzz import process, fuzz

hotels = pd.read_csv("hotels/hotels.csv")

with open("cities.json", "r", encoding="utf-8") as file:
    cities = json.load(file)

cities_df = pd.DataFrame(cities)

def normalize_city(city):
    if pd.isna(city):
        return ""

    city = str(city).lower().strip()
    city = unicodedata.normalize("NFKD", city)
    city = "".join(c for c in city if not unicodedata.combining(c))

    return city

hotels["city_normalized"] = hotels["city"].apply(normalize_city)
cities_df["city_normalized"] = cities_df["name"].apply(normalize_city)

city_map = dict(zip(cities_df["city_normalized"], cities_df["id"]))

hotels["city_id"] = hotels["city_normalized"].map(city_map)

unmatched = hotels[hotels["city_id"].isna()].copy()

city_names = cities_df["city_normalized"].tolist()

print("========== FUZZY CITY MATCHING ==========")

for city in sorted(unmatched["city_normalized"].unique()):
    result = process.extractOne(
        city,
        city_names,
        scorer=fuzz.ratio
    )

    print(city, "->", result)

========== FUZZY CITY MATCHING ==========
 -> ('mumbai', 0.0, 0)
abdullapuram -> ('malappuram', 72.72727272727273, 328)
abu road -> ('habra', 61.53846153846154, 230)
adur -> ('tandur', 80.0, 486)
agonda -> ('nagda', 72.72727272727273, 332)
aguada -> ('nagda', 72.72727272727273, 332)
ahar -> ('lahar', 88.88888888888889, 794)
ahmadnagar -> ('ahmednagar', 90.0, 105)
ajanta -> ('anantnag', 71.42857142857143, 304)
alibaug -> ('hazaribag', 62.5, 241)
alipur -> ('lalitpur', 85.71428571428572, 251)
alleppey -> ("o' valley", 58.82352941176471, 1123)
almora -> ('amroha', 66.66666666666667, 45)
alto porvorim -> ('taraori', 50.0, 1002)
ambalapulai -> ('amalapuram', 76.19047619047619, 584)
ambalavayal -> ('malavalli', 70.0, 792)
angamali -> ('sangli', 71.42857142857143, 76)
anjuna -> ('anjar', 72.72727272727273, 378)
arambol -> ('baramula', 66.66666666666667, 200)
arpora -> ('warora', 83.33333333333334, 657)
attibele -> ('attingal', 62.5, 766)
attigundi -> ('pattamundai', 70.0, 851)
auroville -> ('

In [96]:
city_aliases = {
    "bangalore": "bengaluru",
    "cochin": "kochi",
    "calcutta": "kolkata",
    "trivandrum": "thiruvananthapuram",
    "alleppey": "alappuzha",
    "ooty": "udhagamandalam",
    "baroda": "vadodara",
    "pondicherry": "puducherry",
    "benaras": "varanasi",
    "benares": "varanasi",
    "ahmadnagar": "ahmednagar"
}

In [97]:
hotels["canonical_city"] = hotels["city_normalized"].replace(city_aliases)

hotels["city_id"] = hotels["canonical_city"].map(
    dict(zip(cities_df["city_normalized"], cities_df["id"]))
)

In [98]:
print("Total hotels:", len(hotels))
print("Matched:", hotels["city_id"].notna().sum())
print("Unmatched:", hotels["city_id"].isna().sum())

Total hotels: 6000
Matched: 3764
Unmatched: 2236


In [99]:
alias_rows = hotels[hotels["city_normalized"].isin(city_aliases)]

print(alias_rows[
    ["city", "canonical_city", "state", "city_id"]
].drop_duplicates().to_string(index=False))

       city     canonical_city       state city_id
       ooty     udhagamandalam  Tamil Nadu     371
  bangalore          bengaluru   Karnataka       3
   alleppey          alappuzha      Kerala     193
     cochin              kochi      Kerala      68
 trivandrum thiruvananthapuram      Kerala      58
pondicherry         puducherry Pondicherry     NaN
 ahmadnagar         ahmednagar Maharashtra     106


In [100]:
print("Exact matches:", exact_match.sum())
print("After verified aliases:", hotels["city_id"].notna().sum())
print("Still unmatched:", hotels["city_id"].isna().sum())

Exact matches: 3375
After verified aliases: 3764
Still unmatched: 2236


In [101]:
unmatched = hotels[hotels["city_id"].isna()]

print("Unmatched unique cities:", unmatched["canonical_city"].nunique())
print("\nTop unmatched cities:")
print(unmatched["canonical_city"].value_counts().head(30))

Unmatched unique cities: 390

Top unmatched cities:
canonical_city
jaisalmer         79
calangute         62
dharamshala       41
candolim          38
pushkar           37
kodaikanal        37
baga              36
puducherry        34
palolem           33
lonavala          32
haridwar          31
munnar            31
anjuna            30
thekkady          29
dalhousie         29
bhubaneshwar      28
kumarakom         27
katra             24
kolhapur          22
mandrem           20
darjeeling        20
salt lake city    20
varca             18
                  18
gangtok           17
ghaziabad         17
vagator           17
khajuraho         16
greater noida     16
mcleod ganj       16
Name: count, dtype: int64


In [102]:
cleaned_hotels = hotels[
    hotels["city_id"].notna() &
    hotels["property_name"].notna() &
    hotels["property_id"].notna()
].copy()

print("Rows:", len(cleaned_hotels))

Rows: 3394


In [103]:
cleaned_hotels["site_review_count"] = pd.to_numeric(
    cleaned_hotels["site_review_count"],
    errors="coerce"
)

In [104]:
print(cleaned_hotels["site_review_count"].dtype)

float64


In [105]:
cleaned_hotels["hotel_star_rating"] = (
    cleaned_hotels["hotel_star_rating"]
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

In [106]:
city_names = set(cities_df["city_normalized"])

for old, new in city_aliases.items():
    if new in city_names:
        print(old, "->", new)
    else:
        print("NOT FOUND:", old, "->", new)

bangalore -> bengaluru
cochin -> kochi
calcutta -> kolkata
trivandrum -> thiruvananthapuram
alleppey -> alappuzha
ooty -> udhagamandalam
baroda -> vadodara
NOT FOUND: pondicherry -> puducherry
benaras -> varanasi
benares -> varanasi
ahmadnagar -> ahmednagar


In [107]:
hotels[["city", "state"]].head(20)

,city,state
0,pauri,Uttarakhand
1,ooty,Tamil Nadu
2,pīpalkoti,Uttarakhand
3,kumarakom,Kerala
4,kumarakom,Kerala
5,ooty,Tamil Nadu
6,manāli,Himachal Pradesh
7,haridwār,Uttarakhand
8,kumarakom,Kerala
9,pura raghunāth,Uttar Pradesh


In [108]:
cities_df.head(20)

,id,name,state,city_normalized
0,1,Mumbai,Maharashtra,mumbai
1,2,Delhi,Delhi,delhi
2,3,Bengaluru,Karnataka,bengaluru
3,4,Ahmedabad,Gujarat,ahmedabad
4,5,Hyderabad,Telangana,hyderabad
5,6,Chennai,Tamil Nadu,chennai
6,7,Kolkata,West Bengal,kolkata
7,8,Pune,Maharashtra,pune
8,9,Jaipur,Rajasthan,jaipur
9,10,Surat,Gujarat,surat


In [109]:
print("Hotels with city:", hotels["city"].notna().sum())
print("Hotels with state:", hotels["state"].notna().sum())
print("Cities with state:", cities_df["state"].notna().sum())

Hotels with city: 5982
Hotels with state: 5951
Cities with state: 1221


In [110]:
exact_match = hotels["city_normalized"].isin(set(cities_df["city_normalized"]))

print("Exact matched hotels:", exact_match.sum())
print("Exact unmatched hotels:", (~exact_match).sum())

Exact matched hotels: 3375
Exact unmatched hotels: 2625


In [111]:
cleaned_hotels["hotel_star_rating"] = (
    cleaned_hotels["hotel_star_rating"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)
print(cleaned_hotels["hotel_star_rating"].value_counts(dropna=False).sort_index())

hotel_star_rating
1.0     169
2.0     422
3.0     816
4.0     166
5.0     114
NaN    1707
Name: count, dtype: int64


In [112]:
stay_data = cleaned_hotels[
    [
        "property_id",
        "city_id",
        "property_name",
        "property_type",
        "hotel_star_rating",
        "site_review_rating",
        "site_review_count",
        "hotel_facilities",
        "latitude",
        "longitude"
    ]
].copy()

In [113]:
stay_data["stay_id"] = stay_data["property_id"].astype(int).astype(str)

In [114]:
stay_data = stay_data[
    [
        "stay_id",
        "city_id",
        "property_name",
        "property_type",
        "hotel_star_rating",
        "site_review_rating",
        "site_review_count",
        "hotel_facilities",
        "latitude",
        "longitude"
    ]
]

In [115]:
print("STAY rows:", len(stay_data))
print("STAY columns:", len(stay_data.columns))
print(stay_data.head())

STAY rows: 3394
STAY columns: 10
    stay_id city_id                 property_name property_type  \
0   1742658    1035                Khirsu By GMVN           204   
1   1755986     371           Global Village Ooty           212   
5   1756349     371                 The Hill Side           201   
6   1737428    1218           Hotel Mountain Face           204   
10  1735523     696  The Gaff in Kalimpong-Deluxe           201   

    hotel_star_rating  site_review_rating  site_review_count  \
0                 NaN                 NaN                NaN   
1                 3.0                 7.6                5.0   
5                 NaN                 8.4               81.0   
6                 2.0                 6.9               16.0   
10                NaN                 9.0               51.0   

                                     hotel_facilities   latitude  longitude  
0   Bathroom:Toilet paper|Linen|Towels|Bathroom|To...  30.123749  78.793774  
1   Bathroom:Toilet pap

In [116]:
city_data = cities_df[
    ["id", "name", "state"]
].copy()
city_data.columns = ["city_id", "city_name", "state"]

In [117]:
print("CITY rows:", len(city_data))
print("CITY columns:", len(city_data.columns))
print(city_data.head())

CITY rows: 1221
CITY columns: 3
  city_id  city_name        state
0       1     Mumbai  Maharashtra
1       2      Delhi        Delhi
2       3  Bengaluru    Karnataka
3       4  Ahmedabad      Gujarat
4       5  Hyderabad    Telangana


In [118]:
city_data.to_csv("city_data.csv", index=False)
print("city_data.csv created")

city_data.csv created


In [119]:
print("HOTEL DATASET")
print("Rows:", len(hotels))
print("Columns:", len(hotels.columns))

print("\nMissing property name:", hotels["property_name"].isna().sum())
print("Missing property ID:", hotels["property_id"].isna().sum())
print("Missing city:", hotels["city"].isna().sum())
print("\nCITY MAPPING")
print("Exact + alias mapped:", hotels["city_id"].notna().sum())
print("Unmapped:", hotels["city_id"].isna().sum())

HOTEL DATASET
Rows: 6000
Columns: 32

Missing property name: 590
Missing property ID: 590
Missing city: 18

CITY MAPPING
Exact + alias mapped: 3764
Unmapped: 2236


In [120]:
print("Total:", len(hotels))

print("\nBoth property name and ID present:",
      hotels["property_name"].notna().sum())

print("City mapped:",
      hotels["city_id"].notna().sum())

print("Property + city mapped:",
      (hotels["property_name"].notna() &
       hotels["property_id"].notna() &
       hotels["city_id"].notna()).sum())

Total: 6000

Both property name and ID present: 5410
City mapped: 3764
Property + city mapped: 3394


In [121]:
print("\nUnmapped hotels with property information:")

unmapped = hotels[
    hotels["city_id"].isna() &
    hotels["property_name"].notna() &
    hotels["property_id"].notna()
]

print("Count:", len(unmapped))
print(
    unmapped[
        ["property_name", "city", "state", "latitude", "longitude"]
    ].head(20).to_string(index=False)
)


Unmapped hotels with property information:
Count: 2016
            property_name         city       state  latitude  longitude
            TRH Pipalkoti    pīpalkoti Uttarakhand 30.429540  79.428116
      Swasti house boat 2    kumarakom      Kerala  9.616057  76.428269
     Amrutham Houseboat 2    kumarakom      Kerala  9.632854  76.423570
      Rahi Motel Haridwar     haridwār Uttarakhand 29.946959  78.156489
      Swasti house boat 4    kumarakom      Kerala  9.617157  76.427840
          Oceana Homestay   tiruvallam      Kerala  8.427137  76.961082
              Grand Hotel     haridwār Uttarakhand 29.937243  78.139983
   Center Point Residency  ganpatipule Maharashtra 17.149583  73.273301
Mistletoe Homestay & Cafe       munnar      Kerala 10.018091  77.010495
  Harithavanam Farm House vandiperiyār      Kerala  9.612581  77.069525
           Hotel Premdeep     mātherān Maharashtra 18.985336  73.269796
     The Clive and Curzon        ketti  Tamil Nadu 11.362699  76.728996
        

In [122]:
stay_data.to_csv("stay_data.csv", index=False)
print("stay_data.csv created")

stay_data.csv created
